# Test complet Qwen3.6-27B-FP8

PDF → images → chat multimodal → génération → JSON.

**Avant exécution :** vérifier `MODEL_PATH` et `PDF_PATH` dans la cellule de configuration.


In [ ]:
# ============================================================
# TEST COMPLET QWEN3.6-27B-FP8
# PDF -> IMAGES -> CHAT MULTIMODAL -> GENERATION -> JSON
#
# Objectif :
#   tester la chaîne Qwen3.6 native AVANT intégration pipeline
# ============================================================

import os
import json
import time
import traceback

import torch
import pymupdf
from PIL import Image

from transformers import AutoProcessor, AutoModelForMultimodalLM


# ============================================================


In [ ]:
# 1. CONFIGURATION
# ============================================================

MODEL_PATH = (
    "/domino/edv/modelhub/"
    "ModelHub-model-huggingface-Qwen/"
    "Qwen3.6-27B-FP8/main"
)

# >>> A MODIFIER SI BESOIN <<<
PDF_PATH = "/mnt/data/transferts/OZTURK HASAN.pdf"

# On commence avec UNE seule page.
# Si elle fonctionne, mettre 2 puis 4.
MAX_PAGES_TEST = 1

# Résolution raisonnable pour le premier test.
PDF_DPI = 150

# Suffisant pour voir si le modèle produit un JSON correct.
MAX_NEW_TOKENS = 1200


print("=" * 90)
print("TEST QWEN3.6-27B-FP8")
print("=" * 90)

print("Torch       :", torch.__version__)

try:
    import transformers
    print("Transformers:", transformers.__version__)
except Exception:
    pass

try:
    import kernels
    print("Kernels     :", kernels.__version__)
except Exception as e:
    print("Kernels     : erreur ->", e)

print("CUDA dispo  :", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU         :", torch.cuda.get_device_name(0))
    print(
        "VRAM totale:",
        round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
        "GB"
    )
    print(
        "VRAM libre :",
        round(torch.cuda.mem_get_info()[0] / 1024**3, 2),
        "GB"
    )

print()


# ============================================================


In [ ]:
# 2. VERIFICATIONS FICHIERS
# ============================================================

if not os.path.isdir(MODEL_PATH):
    raise FileNotFoundError(
        f"MODEL_PATH introuvable :\n{MODEL_PATH}"
    )

if not os.path.isfile(PDF_PATH):
    raise FileNotFoundError(
        f"PDF introuvable :\n{PDF_PATH}"
    )

print("MODEL_PATH OK")
print("PDF_PATH   OK")
print()


# ============================================================


In [ ]:
# 3. CHARGEMENT PROCESSOR
# ============================================================

print("=" * 90)
print("CHARGEMENT PROCESSOR")
print("=" * 90)

t0 = time.time()

processor = AutoProcessor.from_pretrained(
    MODEL_PATH,
    local_files_only=True,
    trust_remote_code=True,
)

print("Processor :", processor.__class__.__name__)
print(f"Chargé en : {time.time() - t0:.1f}s")
print()


# ============================================================


In [ ]:
# 4. CHARGEMENT MODELE
# ============================================================

print("=" * 90)
print("CHARGEMENT MODELE")
print("=" * 90)

t0 = time.time()

model = AutoModelForMultimodalLM.from_pretrained(
    MODEL_PATH,

    # IMPORTANT :
    # on laisse Transformers exploiter la quantification FP8
    # enregistrée dans le modèle.
    device_map="auto",

    local_files_only=True,
    trust_remote_code=True,

    low_cpu_mem_usage=True,
)

model.eval()

print("Classe réelle :", model.__class__.__name__)
print("Device        :", next(model.parameters()).device)
print(f"Chargé en     : {time.time() - t0:.1f}s")

if torch.cuda.is_available():
    print(
        "VRAM allouée :",
        round(torch.cuda.memory_allocated(0) / 1024**3, 2),
        "GB"
    )
    print(
        "VRAM réservée:",
        round(torch.cuda.memory_reserved(0) / 1024**3, 2),
        "GB"
    )
    print(
        "VRAM libre   :",
        round(torch.cuda.mem_get_info()[0] / 1024**3, 2),
        "GB"
    )

print()


# ============================================================


In [ ]:
# 5. PDF -> PIL IMAGES
# ============================================================

print("=" * 90)
print("CONVERSION PDF -> IMAGES")
print("=" * 90)

doc = pymupdf.open(PDF_PATH)

print("Nombre total de pages PDF :", len(doc))

n_pages = min(len(doc), MAX_PAGES_TEST)

images = []

zoom = PDF_DPI / 72.0
matrix = pymupdf.Matrix(zoom, zoom)

for page_number in range(n_pages):

    page = doc.load_page(page_number)

    pix = page.get_pixmap(
        matrix=matrix,
        alpha=False
    )

    img = Image.frombytes(
        "RGB",
        [pix.width, pix.height],
        pix.samples
    )

    images.append(img)

    print(
        f"Page {page_number + 1}: "
        f"{img.width} x {img.height}"
    )

doc.close()

print(f"\n{len(images)} image(s) prête(s)")
print()


# ============================================================


In [ ]:
# 6. PROMPT
# ============================================================

PROMPT = """
Tu analyses un dossier bancaire de transfert international.

Lis attentivement l'intégralité du document fourni.

Tu dois extraire les informations réellement visibles dans le document.
N'invente aucune information.
Si une information n'est pas présente ou n'est pas lisible, utilise null.

Retourne UNIQUEMENT un objet JSON valide.

Structure attendue :

{
  "type_document": null,
  "nom_client": null,
  "prenom_client": null,
  "raison_sociale": null,
  "numero_compte": null,
  "devise": null,
  "montant": null,
  "beneficiaire": null,
  "banque_beneficiaire": null,
  "iban": null,
  "swift_bic": null,
  "motif_transfert": null,
  "date_document": null,
  "reference": null,
  "texte_principal": null
}

Pour texte_principal, restitue les principales informations textuelles
lisibles permettant de contrôler la qualité de lecture du document.

Aucun commentaire avant ou après le JSON.
"""


# ============================================================


In [ ]:
# 7. CONSTRUCTION MESSAGE MULTIMODAL QWEN3.6
# ============================================================

print("=" * 90)
print("CONSTRUCTION DU MESSAGE MULTIMODAL")
print("=" * 90)

content = []

for i, image in enumerate(images):

    # IMPORTANT :
    # Qwen3.6 reçoit l'image DANS le message multimodal.
    content.append({
        "type": "image",
        "image": image
    })

content.append({
    "type": "text",
    "text": PROMPT
})

messages = [
    {
        "role": "user",
        "content": content
    }
]

print("Nombre d'images dans le message :", len(images))
print("Message construit.")
print()


# ============================================================


In [ ]:
# 8. PROCESSING NATIF QWEN3.6
# ============================================================

print("=" * 90)
print("PROCESSOR.APPLY_CHAT_TEMPLATE")
print("=" * 90)

try:

    inputs = processor.apply_chat_template(
        messages,

        # POINT ESSENTIEL
        tokenize=True,

        add_generation_prompt=True,

        # POINT ESSENTIEL
        return_dict=True,

        # POINT ESSENTIEL
        return_tensors="pt",
    )

except Exception:

    print("\nERREUR APPLY_CHAT_TEMPLATE")
    traceback.print_exc()
    raise


print("\nClés produites par le processor :")

for key in inputs.keys():

    value = inputs[key]

    if hasattr(value, "shape"):
        print(
            f"  {key:25s}",
            tuple(value.shape),
            value.dtype
        )
    else:
        print(
            f"  {key:25s}",
            type(value)
        )


# ============================================================


In [ ]:
# 9. VERIFICATION CRITIQUE VISION
# ============================================================

print("\n" + "=" * 90)
print("VERIFICATION DES INPUTS VISION")
print("=" * 90)

vision_keys = [
    "pixel_values",
    "image_grid_thw",
]

for k in vision_keys:

    if k in inputs:

        v = inputs[k]

        print(
            f"{k}: PRESENT",
            tuple(v.shape) if hasattr(v, "shape") else type(v)
        )

    else:

        print(f"{k}: ABSENT")


# Pour un vrai input image Qwen multimodal,
# pixel_values doit normalement être présent.

if "pixel_values" not in inputs:

    raise RuntimeError(
        "\nERREUR CRITIQUE : pixel_values absent.\n"
        "Le modèle ne reçoit donc probablement PAS l'image.\n"
        "On arrête ici avant generate()."
    )


# ============================================================


In [ ]:
# 10. ENVOI VERS LE DEVICE
# ============================================================

model_device = next(model.parameters()).device

print("\nDevice modèle :", model_device)

inputs = inputs.to(model_device)

print("Inputs transférés sur le device.")
print()


# ============================================================


In [ ]:
# 11. GENERATION
# ============================================================

print("=" * 90)
print("GENERATION QWEN3.6")
print("=" * 90)

input_length = inputs["input_ids"].shape[-1]

print("Tokens input :", input_length)

t0 = time.time()

with torch.inference_mode():

    generated_ids = model.generate(
        **inputs,

        max_new_tokens=MAX_NEW_TOKENS,

        # OCR / extraction structurée :
        # génération déterministe pour ce test
        do_sample=False,

        use_cache=True,
    )

generation_time = time.time() - t0

print(f"Génération terminée en {generation_time:.1f}s")
print()


# ============================================================


In [ ]:
# 12. SUPPRESSION DES TOKENS DU PROMPT
# ============================================================

generated_only = generated_ids[:, input_length:]

print(
    "Tokens générés :",
    generated_only.shape[-1]
)


# ============================================================


In [ ]:
# 13. DECODAGE
# ============================================================

raw_output = processor.batch_decode(
    generated_only,

    skip_special_tokens=True,

    # IMPORTANT pour éviter certaines modifications
    # du texte généré
    clean_up_tokenization_spaces=False,

)[0]


print("\n")
print("=" * 90)
print("SORTIE BRUTE EXACTE DU MODELE")
print("=" * 90)
print(raw_output)
print("=" * 90)


# ============================================================


In [ ]:
# 14. TEST SIMPLE ANTI-BOUCLE
# ============================================================

print("\nDIAGNOSTIC SORTIE")

print("Longueur caractères :", len(raw_output))

if len(raw_output) == 0:

    print("❌ SORTIE VIDE")

else:

    print("✅ Sortie non vide")


# quelques symptômes de la sortie précédente
bad_patterns = [
    "/sec/sec/sec",
    "Qual Qual Qual",
]

for pattern in bad_patterns:

    if pattern in raw_output:

        print(
            f"❌ Pattern anormal détecté : {pattern}"
        )


# ============================================================


In [ ]:
# 15. EXTRACTION JSON
# ============================================================

def extract_json(text):

    if not text:
        return None

    text = text.strip()

    # retire éventuellement ```json ... ```
    if text.startswith("```"):

        lines = text.splitlines()

        if lines:
            lines = lines[1:]

        if lines and lines[-1].strip().startswith("```"):
            lines = lines[:-1]

        text = "\n".join(lines).strip()

    # premier essai : sortie entière
    try:
        return json.loads(text)

    except Exception:
        pass

    # deuxième essai :
    # recherche du premier { et dernier }
    start = text.find("{")
    end = text.rfind("}")

    if start >= 0 and end > start:

        candidate = text[start:end + 1]

        try:
            return json.loads(candidate)

        except Exception as e:

            print(
                "JSON trouvé mais invalide :",
                repr(e)
            )

    return None


data = extract_json(raw_output)


# ============================================================


In [ ]:
# 16. RESULTAT FINAL
# ============================================================

print("\n")
print("=" * 90)
print("RESULTAT JSON")
print("=" * 90)

if data is None:

    print("❌ JSON NON PARSE")

    print(
        "\nIMPORTANT : ne pas conclure que le PDF "
        "n'a pas été lu."
    )

    print(
        "Regarder d'abord la SORTIE BRUTE ci-dessus."
    )

else:

    print("✅ JSON VALIDE\n")

    print(
        json.dumps(
            data,
            ensure_ascii=False,
            indent=2
        )
    )


# ============================================================


In [ ]:
# 17. BILAN TECHNIQUE
# ============================================================

print("\n")
print("=" * 90)
print("BILAN")
print("=" * 90)

print("Classe modèle :", model.__class__.__name__)
print("Processor     :", processor.__class__.__name__)
print("Pages testées :", len(images))
print("Tokens input  :", input_length)
print("Tokens output :", generated_only.shape[-1])
print("Temps         :", round(generation_time, 1), "s")
print("JSON valide   :", data is not None)

if torch.cuda.is_available():

    print(
        "VRAM allouée :",
        round(
            torch.cuda.memory_allocated(0) / 1024**3,
            2
        ),
        "GB"
    )

    print(
        "VRAM réservée:",
        round(
            torch.cuda.memory_reserved(0) / 1024**3,
            2
        ),
        "GB"
    )

print("=" * 90)
